# Territorial Typology Integration: Goerlich et al. (2016)
## *RURIMESCAPE — Paper 1, Step 0*

This notebook integrates the rural/urban territorial typology from **Goerlich, Reig & Cantarino (2016)**
into the project's demographic dataset. The resulting column `tipo_goerlich` is the stratification
backbone for all Paper 1 analyses.

It also joins the typology with municipal geometries (IGN shapefiles) to produce a
GeoPackage suitable for spatial analysis and visualization.

**Source:** Goerlich, F.J., Reig, E. & Cantarino, I. (2016). *Construcción de una tipología rural/urbana
para los municipios españoles*. FBBVA. Dataset: https://doi.org/10.5281/zenodo.11110545

| | File |
|---|---|
| **Input 1** | `01_padron_clean_1996_2025.csv` |
| **Input 2** | `Reig__Goerlich___Cantarino__2016__FBBVA-IT_Delimitacion_Areas_Rurales.xlsx` |
| **Input 3** | mun_geographic_administrative_hierarchy.gpkg (geometries) |
| **Output 1** | `p0_padron_goerlich_1996_2025.csv` — demographic time series + typology |
| **Output 2** | `p0_municipios_goerlich.gpkg` — one row per municipality, geometry + typology |

In [ ]:
"""
Notebook: p0_goerlich_typology_integration.ipynb
Author: Juan Zotes
Created: 2026-03

Purpose:
    Integrate Goerlich et al. (2016) rural/urban typology into the padrón dataset.
    Diagnose and resolve municipal code mismatches between the two sources.
    Produce a GeoPackage combining geometry, typology, and 2025 population.

Inputs:
    - 01_padron_clean_1996_2025.csv   → Mun_Code as 5-char zero-padded string
    - Goerlich 2016 xlsx              → INECodMuni as integer (2011 census boundaries)
    - mun_geographic_administrative_hierarchy.gpkg        → geometries (ETRS89 geographic, EPSG:4258)

Outputs:
    - p0_padron_goerlich_1996_2025.csv    (sep=';', encoding='utf-8')
    - p0_municipios_goerlich.gpkg

Boundary discrepancy:
    Padrón 2025: 8,132 municipalities (current boundaries).
    Goerlich (2016): 8,116 municipalities (2011 census boundaries).
    Difference: 20 municipalities absent from Goerlich, all created after 2011:
        - 18 segregations: new municipalities split off from an existing parent.
          Typology inherited from parent municipality (Section 6.6).
        - 2 fusions: Oza-Cesuras (15902) and Cerdedo-Cotobade (36902), created
          by merging two municipalities that both shared the same typology.
          Typology inherited unambiguously (Section 6.6).
    After assignment: 100% coverage of 8,132 municipalities.
    See Section 6 for full diagnostic and INE alteration records.

Status: Initial implementation
"""

## 1. Imports and paths

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [ ]:
# Paths
BASE_DIR       = Path(r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain")

DEMOGRAPHY_DIR = BASE_DIR / "data" / "demography" / "processed"
TYPOLOGY_DIR   = BASE_DIR / "data" / "typology" / "raw"
SPATIAL_DIR    = BASE_DIR / "data" / "spatial" / "derived"       # gpkg location
OUTPUT_DIR     = BASE_DIR / "data" / "demography" / "processed"
SPATIAL_OUT    = BASE_DIR / "data" / "spatial" / "processed"

# Create dirs if they don't exist yet
for d in [TYPOLOGY_DIR, SPATIAL_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# Input files
PADRON_FILE    = DEMOGRAPHY_DIR / "01_padron_clean_1996_2025.csv"
GOERLICH_FILE  = TYPOLOGY_DIR  / "Reig, Goerlich & Cantarino (2016)_FBBVA-IT_Delimitacion_Areas_Rurales.xlsx"

# Municipal GeoPackage (peninsula + Canarias)
# Source: IGN — mun_geographic_administrative_hierarchy.gpkg
GPKG_FILE = SPATIAL_DIR   / "mun_geographic_administrative_hierarchy.gpkg"

## 2. Load data

Two datasets with different key formats:
- **Padrón** — `Mun_Code` is a zero-padded **string** (`'01001'`)
- **Goerlich** — `INECodMuni` is an **integer** stored as string (`'1001'` for Álava provinces)

We normalize both to a zero-padded 5-character string before merging.

In [ ]:
# Load padrón
padron = pd.read_csv(
    PADRON_FILE,
    sep=",",             # adjust if your file uses comma
    dtype={"Mun_Code": str}
)

print(f"Padrón rows:           {len(padron):,}")
print(f"Municipalities:        {padron['Mun_Code'].nunique():,}")
print(f"Years:                 {sorted(padron['Year'].unique())}")
padron.head(3)

In [ ]:
# Load Goerlich typology
goerlich = pd.read_excel(
    GOERLICH_FILE,
    sheet_name="Tipologia_Final",
    dtype={"INECodMuni": str}
)

print(f"Goerlich rows:         {len(goerlich):,}")
print(f"Columns:               {list(goerlich.columns)}")
goerlich.head(3)

## 3. Harmonize municipality codes

Goerlich's `INECodMuni` lacks leading zeros for provinces 01–09 (Álava, Albacete, Alicante…).
A simple `zfill(5)` aligns both datasets.

In [ ]:
goerlich["Mun_Code"] = goerlich["INECodMuni"].str.zfill(5)

# Sanity check
assert (goerlich["Mun_Code"].str.len() == 5).all(), "Some Goerlich codes are not 5 characters!"
assert (padron["Mun_Code"].str.len() == 5).all(),   "Some padrón codes are not 5 characters!"

print("Code length check passed ✓")
print("Goerlich sample:", goerlich["Mun_Code"].head(5).tolist())
print("Padrón sample:  ", padron["Mun_Code"].head(5).tolist())

## 4. Select typology columns

The Goerlich dataset contains several intermediate classification variables
(`Tipologia`, `TipologiaLC`, `Proximity45`, `LC`). For this project we retain
only the **final typology** column (`Final`), renamed to `tipo_goerlich`.

The six categories are:

| `tipo_goerlich` | Description |
|---|---|
| `Urbano - Abierto` | Urban, open land cover |
| `Urbano - Cerrado` | Urban, closed land cover |
| `Intermedio - Abierto` | Intermediate, open land cover |
| `Intermedio - Cerrado` | Intermediate, closed land cover |
| `Rural - Accesible` | Rural, within 45 min of urban centre |
| `Rural - Remoto` | Rural, beyond 45 min of urban centre |


In [ ]:
goerlich_slim = (
    goerlich[["Mun_Code", "Final"]]
    .rename(columns={"Final": "tipo_goerlich"})
)

print("Typology distribution in Goerlich dataset:")
print(goerlich_slim["tipo_goerlich"].value_counts().to_string())

## 5. Merge padrón + typology

**Left join** on `Mun_Code`: keeps all padrón rows (all municipalities × all years).
Municipalities absent from Goerlich receive `NaN` in `tipo_goerlich` — diagnosed in Section 6.

In [ ]:
padron_typed = padron.merge(
    goerlich_slim,
    on="Mun_Code",
    how="left"
)

print(f"Shape after merge: {padron_typed.shape}")
padron_typed.head(3)

## 6. Diagnose municipal code mismatches

The padrón covers municipalities as of 2025; Goerlich uses 2012 boundaries.
We expect a small discrepancy (~16 municipalities) due to:
- **Fusions post-2012**: two or more small municipalities merged into one → the new code
  appears in the padrón but not in Goerlich (which has both old codes).
- **Segregations post-2012**: a municipality split → new code only in padrón.
- **Canary Islands / Ceuta / Melilla**: Goerlich may have excluded some territories.

Understanding *why* each municipality is unmatched is important before deciding
how to handle them in the analysis (exclude, impute from parent, or keep as `NaN`).

In [ ]:
# --- 6.1  Municipalities in padrón but NOT in Goerlich ---
padron_codes   = set(padron["Mun_Code"].unique())
goerlich_codes = set(goerlich_slim["Mun_Code"].unique())

only_in_padron = padron_codes - goerlich_codes

unmatched = (
    padron[padron["Mun_Code"].isin(only_in_padron)]
    [["Mun_Code", "Mun"]]
    .drop_duplicates()
    .sort_values("Mun_Code")
    .reset_index(drop=True)
)

print(f"Municipalities in padrón:          {len(padron_codes):,}")
print(f"Municipalities in Goerlich:        {len(goerlich_codes):,}")
print(f"Only in padrón (unmatched):        {len(only_in_padron)}")
print()
print("Full list of unmatched municipalities:")
unmatched

In [ ]:
# --- 6.2  Municipalities in Goerlich but NOT in padrón ---
# These are municipalities that existed in 2012 but have since been absorbed/fused.
only_in_goerlich = goerlich_codes - padron_codes

dissolved = (
    goerlich[goerlich["Mun_Code"].isin(only_in_goerlich)]
    [["Mun_Code", "NombreMuni"]]
    .merge(goerlich_slim, on="Mun_Code")
    .sort_values("Mun_Code")
    .reset_index(drop=True)
)

print(f"Only in Goerlich (dissolved/fused post-2012): {len(only_in_goerlich)}")
print()
dissolved

In [ ]:
# --- 6.3  Coverage summary ---
n_total   = len(padron_codes)
n_matched = n_total - len(only_in_padron)

print(f"Total municipalities in padrón:    {n_total:,}")
print(f"Matched with Goerlich typology:    {n_matched:,}")
print(f"Unmatched (NaN in tipo_goerlich):  {len(only_in_padron)}")
print(f"Coverage:                          {n_matched / n_total * 100:.2f}%")

### 6.4 Note on unmatched municipalities

All 20 unmatched municipalities have been fully identified and documented (Section 6.5).
They fall into two categories:

- **18 segregations**: municipalities created after 2011 by splitting off from an existing
  municipality. They inherit the typology of their parent municipality.
- **2 fusions**: Oza-Cesuras (15902) and Cerdedo-Cotobade (36902), created by merging
  two municipalities that existed in Goerlich. Both parent pairs share the same typology
  (`Rural - Accesible`), making the assignment unambiguous.

All 20 will be assigned the typology of their parent municipality in Section 6.6.
Coverage after assignment: **100%** of the 8,132 municipalities in the padrón.

**Note for Paper 1:** the typology of these 20 municipalities is inherited, not directly
classified by Goerlich et al. (2016). This should be noted in the methods section.
The assignment is defensible because the territorial and demographic characteristics
that determine the typology (land cover, accessibility) are unlikely to differ
substantially between a newly created municipality and its parent.

In [ ]:
# --- 6.5  Investigate unmatched municipalities ---
# INE assigns codes with suffix >= 900 (PPMMM where MMM >= 900) to municipalities
# created after the last census boundary revision — either by fusion or segregation.
# The 3 exceptions (18065, 18077, 18106) are municipalities that disappeared before
# 2011 (absorbed by a neighbour) and were re-created by segregation after 2011,
# retaining their historical code rather than receiving a new 9XX suffix.

# Reset to base unmatched list
unmatched = (
    padron[padron["Mun_Code"].isin(only_in_padron)]
    [["Mun_Code", "Mun"]]
    .drop_duplicates()
    .sort_values("Mun_Code")
    .reset_index(drop=True)
)

alterations = {
    "04904": ("Segregación", "Berja",                  "04029", "15/07/2015"),
    "06903": ("Segregación", "Badajoz",                "06015", "02/07/2012"),
    "10904": ("Segregación", "Talayuela",              "10180", "02/09/2013"),
    "10905": ("Segregación", "Talayuela",              "10180", "09/02/2015"),
    "11903": ("Segregación", "Jimena de la Frontera",  "11021", "16/01/2019"),
    "14901": ("Segregación", "Fuente Palmera",         "14030", "03/12/2018"),
    "14902": ("Segregación", "Santaella",              "14060", "03/12/2018"),
    "15902": ("Fusión",      "Cesuras + Oza dos Ríos", "15026+15063", "—"),
    "18065": ("Segregación", "Iznalloz",               "18105", "18/12/2014"),
    "18077": ("Segregación", "Arenas del Rey",         "18020", "16/01/2019"),
    "18106": ("Segregación", "Arenas del Rey",         "18020", "22/10/2015"),
    "18914": ("Segregación", "Pinos Puente",           "18158", "14/04/2014"),
    "18915": ("Segregación", "Iznalloz",               "18105", "27/01/2016"),
    "18916": ("Segregación", "Motril",                 "18140", "16/01/2019"),
    "21902": ("Segregación", "Calañas",                "21017", "16/01/2019"),
    "29903": ("Segregación", "Ronda",                  "29084", "30/01/2015"),
    "29904": ("Segregación", "Ronda",                  "29084", "02/02/2015"),
    "36902": ("Fusión",      "Cerdedo + Cotobade",     "36011+36012", "—"),
    "41904": ("Segregación", "Utrera",                 "41095", "03/12/2018"),
    "48916": ("Segregación", "Galdácano",              "48036", "28/12/2023"),
}

alt_df = pd.DataFrame.from_dict(
    alterations, orient="index",
    columns=["alteration_type", "origin_name", "origin_code", "date"]
).reset_index().rename(columns={"index": "Mun_Code"})

unmatched = unmatched.merge(alt_df, on="Mun_Code", how="left")

print("Unmatched municipalities — alteration type summary:")
print(f"  Segregaciones: {(unmatched['alteration_type'] == 'Segregación').sum()}")
print(f"  Fusiones:      {(unmatched['alteration_type'] == 'Fusión').sum()}")
print()
unmatched[["Mun_Code", "Mun", "alteration_type", "origin_name", "origin_code", "date"]]


In [ ]:
origin_codes = [
    "04029", "06015", "10180", "11021", "14030", "14060",
    "18105", "18020", "18158", "18140", "21017", "29084",
    "41095", "48036"
]

goerlich_slim[goerlich_slim["Mun_Code"].isin(origin_codes)].sort_values("Mun_Code")

In [ ]:
# --- 6.6  Assign typology to all unmatched municipalities ---
#
# All 20 unmatched municipalities are post-2011 creations absent from Goerlich (2016),
# which uses 2011 census boundaries. They fall into two categories:
#
# FUSIONS (2): created by merging two existing municipalities.
#   Both parent pairs share the same typology → unambiguous assignment.
#
# SEGREGATIONS (18): created by splitting off from an existing municipality.
#   They inherit the typology of their parent municipality, as land cover
#   and accessibility characteristics are unlikely to differ substantially
#   from the parent territory.
#
# Exception to note in methods: Guadiana (06903) inherits Urbano - Abierto
#   from Badajoz, though as a small rural segregation it may not match
#   the urban profile. No better alternative without direct classification.

# --- Fusions ---
fusions = {
    "15902": ("Rural - Accesible", "Cesuras (15026) + Oza dos Ríos (15063)"),
    "36902": ("Rural - Accesible", "Cerdedo (36011) + Cotobade (36012)"),
}

# --- Segregations: new_code → (tipologia, parent_name, parent_code) ---
segregations = {
    "04904": ("Intermedio - Abierto",  "Berja",                 "04029"),
    "06903": ("Urbano - Abierto",      "Badajoz",               "06015"),
    "10904": ("Intermedio - Abierto",  "Talayuela",             "10180"),
    "10905": ("Intermedio - Abierto",  "Talayuela",             "10180"),
    "11903": ("Intermedio - Abierto",  "Jimena de la Frontera", "11021"),
    "14901": ("Intermedio - Abierto",  "Fuente Palmera",        "14030"),
    "14902": ("Rural - Accesible",     "Santaella",             "14060"),
    "18065": ("Intermedio - Abierto",  "Iznalloz",              "18105"),
    "18077": ("Rural - Accesible",     "Arenas del Rey",        "18020"),
    "18106": ("Rural - Accesible",     "Arenas del Rey",        "18020"),
    "18914": ("Rural - Accesible",     "Pinos Puente",          "18158"),
    "18915": ("Intermedio - Abierto",  "Iznalloz",              "18105"),
    "18916": ("Intermedio - Abierto",  "Motril",                "18140"),
    "21902": ("Rural - Remoto",        "Calañas",               "21017"),
    "29903": ("Intermedio - Abierto",  "Ronda",                 "29084"),
    "29904": ("Intermedio - Abierto",  "Ronda",                 "29084"),
    "41904": ("Intermedio - Abierto",  "Utrera",                "41095"),
    "48916": ("Intermedio - Abierto",  "Galdácano",             "48036"),
}

# --- Apply fusions ---
for code, (tipologia, origin) in fusions.items():
    padron_typed.loc[padron_typed["Mun_Code"] == code, "tipo_goerlich"] = tipologia

# --- Apply segregations ---
for code, (tipologia, origin_name, origin_code) in segregations.items():
    padron_typed.loc[padron_typed["Mun_Code"] == code, "tipo_goerlich"] = tipologia

# --- Verify all 20 assigned ---
all_codes = list(fusions.keys()) + list(segregations.keys())
assigned = (
    padron_typed[padron_typed["Mun_Code"].isin(all_codes)]
    [["Mun_Code", "Mun", "tipo_goerlich"]]
    .drop_duplicates()
    .sort_values("Mun_Code")
    .reset_index(drop=True)
)

print(f"Typology assigned to all unmatched municipalities ({len(assigned)}/20):")
print(f"  Fusions:      {len(fusions)}")
print(f"  Segregations: {len(segregations)}")
print(f"  NaN remaining: {padron_typed['tipo_goerlich'].isna().sum()}")
print()
assigned

## 7. Typology distribution in matched dataset

Final count of **unique municipalities** per category after the merge.

In [ ]:
typology_counts = (
    padron_typed[padron_typed["tipo_goerlich"].notna()]
    .drop_duplicates(subset="Mun_Code")
    ["tipo_goerlich"]
    .value_counts()
    .rename_axis("tipo_goerlich")
    .reset_index(name="n_municipios")
)

typology_counts["pct"] = (
    typology_counts["n_municipios"] / typology_counts["n_municipios"].sum() * 100
).round(1)

typology_counts

In [ ]:
# Add 2025 population and population share to typology summary
pop_2025 = (
    padron_typed[
        (padron_typed["Year"] == 2025) &
        (padron_typed["Cat"] == "Total")
    ]
    .drop_duplicates(subset="Mun_Code")
    .groupby("tipo_goerlich")["Pop"]
    .sum()
    .reset_index()
    .rename(columns={"Pop": "pop_2025"})
)

typology_counts = typology_counts.merge(pop_2025, on="tipo_goerlich")
typology_counts["pct_pop"] = (
    typology_counts["pop_2025"] / typology_counts["pop_2025"].sum() * 100
).round(1)

typology_counts

In [ ]:
years = [1998, 2008, 2018, 2025]

pop_years = (
    padron_typed[
        (padron_typed["Year"].isin(years)) &
        (padron_typed["Cat"] == "Total")
    ]
    .drop_duplicates(subset=["Mun_Code", "Year"])
    .groupby(["tipo_goerlich", "Year"])["Pop"]
    .sum()
    .reset_index()
    .pivot(index="tipo_goerlich", columns="Year", values="Pop")
)

for y in years:
    pop_years[f"pct_{y}"] = (pop_years[y] / pop_years[y].sum() * 100).round(1)

pop_years

## 8. Export CSV

Demographic time series enriched with `tipo_goerlich`.
Separator `;` for compatibility with Spanish locale tools (Excel, QGIS).

In [ ]:
csv_out = OUTPUT_DIR / "p0_padron_goerlich_1996_2025.csv"

padron_typed.to_csv(
    csv_out,
    sep=";",
    index=False,
    encoding="utf-8"
)

print(f"CSV exported → {csv_out}")
print(f"Shape: {padron_typed.shape}")
print(f"Columns: {list(padron_typed.columns)}")

## 9. Build GeoPackage: geometry + typology

This section joins the Goerlich typology with IGN municipal geometries to produce
a GeoPackage with **one row per municipality** (not per year). This file is the
spatial base for all maps in Paper 1.

It contains:
- Municipal geometry (polygon)
- `tipo_goerlich`
- Population in 2025 (from padrón, for label sizing in maps)


In [ ]:
# Load municipal GeoPackage (peninsula + Canarias together)
# We'll inspect the columns first to identify the municipality code field.
gdf = gpd.read_file(GPKG_FILE)

print(f"GeoDataFrame shape: {gdf.shape}")
print(f"CRS: {gdf.crs}")
print(f"Columns: {list(gdf.columns)}")
gdf.head(3)

In [ ]:
# --- Verify municipality code field ---
# This GeoPackage already contains Mun_Code as a 5-digit zero-padded string.
# No extraction needed — just verify alignment with padrón codes.

print("Sample Mun_Code in GeoPackage:")
print(gdf["Mun_Code"].head(5).tolist())
print(f"\nUnique Mun_Code in GeoPackage: {gdf['Mun_Code'].nunique():,}")
print(f"Unique Mun_Code in padrón:      {len(padron_codes):,}")

# Check for any code length issues
assert (gdf["Mun_Code"].str.len() == 5).all(), "Some GeoPackage codes are not 5 characters!"
print("\nCode length check passed ✓")

In [ ]:
print(padron_typed.columns.tolist())
print(padron_typed["Cat"].unique())

In [ ]:
# --- Extract 2025 population from padrón for map labelling ---
pop_2025 = (
    padron_typed[
        (padron_typed["Year"] == 2025) &
        (padron_typed["Cat"] == "Total")   # adjust if your column/value differs
    ]
    [["Mun_Code", "Mun", "Pop", "tipo_goerlich"]]
    .copy()
)

print(f"Rows for 2025 Total: {len(pop_2025)}")
pop_2025.head(3)

In [ ]:
# --- Merge shapefile + typology + population ---
gdf_typed = gdf.merge(
    pop_2025,
    on="Mun_Code",
    how="left"
)

print(f"GeoDataFrame after merge: {gdf_typed.shape}")
print(f"Municipalities with tipo_goerlich: {gdf_typed['tipo_goerlich'].notna().sum():,}")
print(f"Municipalities without (NaN):      {gdf_typed['tipo_goerlich'].isna().sum()}")
gdf_typed[["Mun_Code", "Mun", "tipo_goerlich", "Pop"]].head(5)

In [ ]:
# --- Reproject to ETRS89 geographic (EPSG:4258) if needed ---
# ETRS89 geographic is a good default for Spain: compatible with QGIS, Folium, and GeoPandas.
# If your shapefile is already in EPSG:4258 or EPSG:4326, skip.

if gdf_typed.crs.to_epsg() != 4258:
    gdf_typed = gdf_typed.to_crs(epsg=4258)
    print(f"Reprojected to EPSG:4258 (ETRS89 geographic)")
else:
    print(f"CRS already EPSG:4258 — no reprojection needed")

print(f"Final CRS: {gdf_typed.crs}")

In [ ]:
# --- Export GeoPackage ---
gpkg_out = SPATIAL_OUT / "p0_municipios_goerlich.gpkg"

gdf_typed.to_file(
    gpkg_out,
    driver="GPKG",
    layer="municipios_goerlich_2025"
)

print(f"GeoPackage exported → {gpkg_out}")
print(f"Layer: municipios_goerlich_2025")
print(f"Features: {len(gdf_typed):,}")
print(f"Columns: {[c for c in gdf_typed.columns if c != 'geometry']}")

In [ ]:
# --- Export CSV (no geometry, no population) ---
# Columns to keep: all administrative and typology fields, drop Pop and geometry
 
cols_csv = [
    "Mun_Code",
    "Mun_Name",
    "Comarca_Code",
    "Comarca_Name",
    "Prov_Code",
    "Prov_Name",
    "CCAA_Code",
    "CCAA_Name",
    "nationalcode",
    "tipo_goerlich",
]
 
csv_out = SPATIAL_OUT / "p0_municipios_goerlich_admin_hierarchy.csv"
 
(
    gdf_typed[cols_csv]
    .to_csv(csv_out, index=False, sep=";", encoding="utf-8-sig")
)
 
print(f"CSV exported → {csv_out}")
print(f"Rows : {len(gdf_typed):,}")
print(f"Columns : {cols_csv}")
 

## 10. Summary

| Output | Content | Use |
|---|---|---|
| `p0_padron_goerlich_1996_2025.csv` | Full time series 1996–2025 + `tipo_goerlich` | All Paper 1 statistical analyses |
| `p0_municipios_goerlich.gpkg` | One row per municipality + geometry + typology + Pop 2025 | Maps, spatial analysis |

**Key findings from the diagnostic (Section 6):**
- The exact number of unmatched municipalities is printed above.
- Municipalities only in Goerlich correspond to pre-2012 units that were
  subsequently fused — their typology is preserved in the Goerlich dataset
  but has no current padrón counterpart.
- Municipalities only in padrón are post-2012 creations or the result of fusions.
  They will be treated as `NaN` in the typology and excluded from stratified analyses.

**Next step:** `p1a, p1b, p1c` — verify and date the 2018
inflection point using Pettitt or Mann-Kendall test on aggregated rural residential balance.